In [ ]:
!pip install biopython
!pip install git+https://github.com/songlab-cal/gpn.git
from transformers import AutoTokenizer, AutoModel
import numpy as np
import torch
import time
import os
from Bio import SeqIO

print(torch.cuda.is_available())
device = "cuda" if torch.cuda.is_available() else "cpu"
print(torch.cuda.device_count())

In [ ]:
# Configuration Parameters
BATCH_SIZE = 4  # Adjust based on your GPU memory (reduce if you get OOM errors)
MAX_SEQUENCE_LENGTH = 2048  # Maximum sequence length for tokenization
GENES_DIRECTORY = "100 Genes"  # Directory containing FASTA files
OUTPUT_BASE_DIR = "."  # Base directory for output folders

print("Configuration:")
print(f"Batch Size: {BATCH_SIZE}")
print(f"Max Sequence Length: {MAX_SEQUENCE_LENGTH}")
print(f"Genes Directory: {GENES_DIRECTORY}")
print(f"Output Base Directory: {OUTPUT_BASE_DIR}")

# Gene Embedding Generation for Yeast Strains

## Overview
This notebook generates embeddings for gene sequences across 1011 yeast strains using multiple pre-trained language models. Each FASTA file in the `100 Genes` directory contains sequences for the same gene across all strains.

## Key Optimizations
1. **Batching**: Processes multiple sequences simultaneously to improve GPU utilization
2. **Memory Management**: Automatic GPU memory cleanup between batches and models
3. **File Organization**: Embeddings are saved in model-specific directories
4. **Error Handling**: Robust error handling to continue processing even if individual batches fail
5. **Progress Tracking**: Detailed progress reporting and memory monitoring

## Output Structure
```
├── model_name_1/
│   ├── gene1_embeddings.txt
│   ├── gene2_embeddings.txt
│   └── ...
├── model_name_2/
│   ├── gene1_embeddings.txt
│   ├── gene2_embeddings.txt
│   └── ...
└── embedding_summary.csv
```

## Configuration Parameters
- `BATCH_SIZE`: Number of sequences to process simultaneously (adjust based on GPU memory)
- `MAX_SEQUENCE_LENGTH`: Maximum sequence length for tokenization
- `GENES_DIRECTORY`: Directory containing FASTA files
- `OUTPUT_BASE_DIR`: Base directory for output folders

## Usage
1. Run the configuration cell to set parameters
2. Install dependencies and check GPU availability
3. Load gene sequences from FASTA files
4. Optionally run a test with a subset of data
5. Execute the main embedding generation loop
6. Use utility functions for post-processing and analysis

In [ ]:
genes = {}  # Changed from list to dictionary

In [ ]:
from Bio import SeqIO
from pathlib import Path

print("Loading gene sequences...")
for filename in Path(GENES_DIRECTORY).glob("*.fasta"):
    sequences = []
    
    for sequence in SeqIO.parse(filename, "fasta"):
        sequences.append({'id': len(sequences) + 1, 'seq': str(sequence.seq)})
    
    gene_name = str(filename).split('\\')[-1].split('_')[0]  # Extract gene name properly
    genes[gene_name] = sequences
    print(f"Loaded {len(sequences)} sequences for gene {gene_name}")

print(f"Total genes loaded: {len(genes)}")
print(f"Sample gene names: {list(genes.keys())[:5]}")

In [ ]:
# Debug: Check loaded data
print(f"Number of genes loaded: {len(genes)}")
if genes:
    sample_gene = list(genes.keys())[0]
    print(f"Sample gene: {sample_gene}")
    print(f"Number of sequences for {sample_gene}: {len(genes[sample_gene])}")
    print(f"First sequence ID: {genes[sample_gene][0]['id']}")
    print(f"First sequence length: {len(genes[sample_gene][0]['seq'])}")
else:
    print("No genes loaded!")

In [ ]:
# Utility functions for monitoring and checking results

def check_gpu_memory():
    """Check GPU memory usage"""
    if torch.cuda.is_available():
        print(f"GPU Memory - Allocated: {torch.cuda.memory_allocated()/1024**3:.2f} GB")
        print(f"GPU Memory - Cached: {torch.cuda.memory_reserved()/1024**3:.2f} GB")
    else:
        print("CUDA not available")

def check_embedding_files():
    """Check which embedding files have been generated"""
    from pathlib import Path
    
    model_names = [model.split('/')[1] if '/' in model else model for model in models]
    
    print("Checking generated embedding files:")
    for model_name in model_names:
        model_dir = Path(model_name)
        if model_dir.exists():
            files = list(model_dir.glob("*_embeddings.txt"))
            print(f"{model_name}: {len(files)} files")
            if len(files) < len(genes):
                missing = len(genes) - len(files)
                print(f"  Missing {missing} files")
        else:
            print(f"{model_name}: Directory not found")

def get_embedding_file_sizes():
    """Get sizes of all embedding files"""
    from pathlib import Path
    import os
    
    model_names = [model.split('/')[1] if '/' in model else model for model in models]
    
    total_size = 0
    for model_name in model_names:
        model_dir = Path(model_name)
        if model_dir.exists():
            for file in model_dir.glob("*_embeddings.txt"):
                size = os.path.getsize(file) / (1024**2)  # MB
                total_size += size
                print(f"{file.name}: {size:.2f} MB")
    
    print(f"Total size: {total_size:.2f} MB")

# Check initial GPU memory
check_gpu_memory()

In [ ]:
# Optional: Test with a small subset first
def test_with_subset(num_genes=2, num_sequences_per_gene=10, num_models=1):
    """
    Test the embedding generation with a small subset of data
    """
    print("Running test with subset...")
    
    # Create test subset
    test_genes = {}
    gene_names = list(genes.keys())[:num_genes]
    
    for gene_name in gene_names:
        test_genes[gene_name] = genes[gene_name][:num_sequences_per_gene]
    
    test_models = models[:num_models]
    
    print(f"Test setup: {len(test_models)} model(s), {len(test_genes)} gene(s)")
    
    # Run test
    for model in test_models:
        for gene_name, sequences in test_genes.items():
            print(f"Testing {model} with {gene_name}...")
            try:
                num_embeddings = generateEmbeddings(model, f"test_{gene_name}", sequences, batch_size=2)
                print(f"✓ Success: Generated {num_embeddings} embeddings")
            except Exception as e:
                print(f"✗ Error: {str(e)}")
    
    print("Test completed!")

# Uncomment the line below to run a test first
# test_with_subset()

In [ ]:
def generateEmbeddings(modelName, gene, sequences, batch_size=None):
    """
    Generate embeddings for sequences with batching for improved efficiency.
    
    Args:
        modelName: HuggingFace model identifier
        gene: Gene name
        sequences: List of sequence dictionaries
        batch_size: Number of sequences to process in each batch (uses global BATCH_SIZE if None)
    """
    if batch_size is None:
        batch_size = BATCH_SIZE
        
    print(f"Loading model: {modelName}")
    tokenizer = AutoTokenizer.from_pretrained(modelName, trust_remote_code=True)
    model = AutoModel.from_pretrained(
        modelName, trust_remote_code=True,
    ).to(device)
    
    model.eval()  # Set to evaluation mode
    embeddings = []
    
    start = time.time()
    print(f"Starting to generate {modelName}'s embeddings for gene {gene} ({len(sequences)} sequences)")
    
    # Process sequences in batches
    for i in range(0, len(sequences), batch_size):
        batch = sequences[i:i + batch_size]
        batch_sequences = [str(seq["seq"]) for seq in batch]
        batch_ids = [seq["id"] for seq in batch]
        
        try:
            # Tokenize batch
            inputs = tokenizer(
                batch_sequences,
                return_tensors="pt",
                padding=True,
                truncation=True,
                max_length=MAX_SEQUENCE_LENGTH,
                return_attention_mask=True
            ).to(device)
            
            with torch.no_grad():
                outputs = model(**inputs)
                
            # Extract embeddings (mean pooling over sequence length)
            attention_mask = inputs['attention_mask']
            token_embeddings = outputs.last_hidden_state
            
            # Apply attention mask and compute mean
            masked_embeddings = token_embeddings * attention_mask.unsqueeze(-1)
            summed_embeddings = masked_embeddings.sum(dim=1)
            sequence_lengths = attention_mask.sum(dim=1, keepdim=True)
            batch_embeddings = summed_embeddings / sequence_lengths
            
            # Convert to numpy and store
            batch_embeddings_np = batch_embeddings.cpu().numpy()
            
            for j, (seq_id, embedding) in enumerate(zip(batch_ids, batch_embeddings_np)):
                embeddings.append({
                    "id": seq_id, 
                    "embedding": embedding.tolist()
                })
            
            print(f"Processed batch {i//batch_size + 1}/{(len(sequences) + batch_size - 1)//batch_size}")
            
            # Clear batch from GPU memory
            del inputs, outputs, token_embeddings, batch_embeddings
            torch.cuda.empty_cache() if torch.cuda.is_available() else None
            
        except Exception as e:
            print(f"Error processing batch {i//batch_size + 1}: {str(e)}")
            # Continue with next batch instead of failing completely
            continue
    
    finish = time.time()
    print(f"Finished generating {modelName}'s embeddings for {gene} in {round(finish - start)} seconds.")
    
    # Create model directory if it doesn't exist
    model_dir = os.path.join(OUTPUT_BASE_DIR, modelName.split('/')[1] if '/' in modelName else modelName)
    os.makedirs(model_dir, exist_ok=True)
    
    # Save embeddings to file
    output_file = os.path.join(model_dir, f"{gene}_embeddings.txt")
    with open(output_file, "w") as f:
        f.write(str(embeddings))
    
    print(f"Saved embeddings to: {output_file}")
    
    # Clear GPU memory
    del model
    del tokenizer
    torch.cuda.empty_cache() if torch.cuda.is_available() else None
    
    return len(embeddings)

In [ ]:
models = [
    "LongSafari/hyenadna-medium-450k-seqlen-hf",
    "LongSafari/hyenadna-large-1m-seqlen-hf",
    "AIRI-Institute/gena-lm-bigbird-base-t2t",
    "songlab/gpn-brassicales",
    "PoetschLab/GROVER",
    "InstaDeepAI/nucleotide-transformer-500m-human-ref",
    "InstaDeepAI/nucleotide-transformer-2.5b-multi-species",
    "InstaDeepAI/nucleotide-transformer-2.5b-1000g",
    #"zhihan1996/DNABERT-S",
    #"zhihan1996/DNABERT-2-117M",
]

In [ ]:
# Main execution loop with improved error handling and progress tracking
total_combinations = len(models) * len(genes)
current_combination = 0

print(f"Starting embedding generation for {len(models)} models and {len(genes)} genes")
print(f"Total combinations to process: {total_combinations}")
print(f"Using batch size: {BATCH_SIZE}")

for model_idx, model in enumerate(models):
    print(f"\n{'='*60}")
    print(f"Processing model {model_idx + 1}/{len(models)}: {model}")
    print(f"{'='*60}")
    
    try:
        for gene_idx, (gene_name, sequences) in enumerate(genes.items()):
            current_combination += 1
            print(f"\nProgress: {current_combination}/{total_combinations}")
            print(f"Gene {gene_idx + 1}/{len(genes)}: {gene_name} ({len(sequences)} sequences)")
            
            try:
                num_embeddings = generateEmbeddings(model, gene_name, sequences)
                print(f"Successfully generated {num_embeddings} embeddings for {gene_name}")
                
                # Check GPU memory after each gene
                if torch.cuda.is_available():
                    memory_allocated = torch.cuda.memory_allocated() / 1024**3
                    if memory_allocated > 8:  # Warning if >8GB allocated
                        print(f"⚠️  High GPU memory usage: {memory_allocated:.2f} GB")
                
            except Exception as e:
                print(f"ERROR: Failed to generate embeddings for {gene_name} with {model}: {str(e)}")
                continue
                
    except Exception as e:
        print(f"ERROR: Failed to process model {model}: {str(e)}")
        continue

print(f"\n{'='*60}")
print("Embedding generation completed!")
print(f"{'='*60}")

# Final summary
print("\nFinal summary:")
check_embedding_files()
get_embedding_file_sizes()

In [ ]:
# Additional utilities for post-processing and analysis

import json
import pickle
import pandas as pd

def convert_embeddings_to_numpy():
    """Convert text embedding files to numpy format for easier loading"""
    import ast
    
    model_names = [model.split('/')[1] if '/' in model else model for model in models]
    
    for model_name in model_names:
        model_dir = Path(model_name)
        if model_dir.exists():
            numpy_dir = model_dir / "numpy"
            numpy_dir.mkdir(exist_ok=True)
            
            for txt_file in model_dir.glob("*_embeddings.txt"):
                try:
                    with open(txt_file, 'r') as f:
                        embeddings = ast.literal_eval(f.read())
                    
                    # Extract embeddings and IDs
                    ids = [item['id'] for item in embeddings]
                    embedding_matrix = np.array([item['embedding'] for item in embeddings])
                    
                    # Save as numpy
                    gene_name = txt_file.stem.replace('_embeddings', '')
                    np.save(numpy_dir / f"{gene_name}_embeddings.npy", embedding_matrix)
                    np.save(numpy_dir / f"{gene_name}_ids.npy", np.array(ids))
                    
                    print(f"Converted {txt_file.name} -> {gene_name}_embeddings.npy")
                    print(f"Shape: {embedding_matrix.shape}")
                    
                except Exception as e:
                    print(f"Error converting {txt_file}: {e}")

def create_summary_report():
    """Create a summary report of all generated embeddings"""
    model_names = [model.split('/')[1] if '/' in model else model for model in models]
    
    summary_data = []
    
    for model_name in model_names:
        model_dir = Path(model_name)
        if model_dir.exists():
            for txt_file in model_dir.glob("*_embeddings.txt"):
                try:
                    with open(txt_file, 'r') as f:
                        content = f.read()
                    
                    # Parse embeddings
                    import ast
                    embeddings = ast.literal_eval(content)
                    
                    gene_name = txt_file.stem.replace('_embeddings', '')
                    num_sequences = len(embeddings)
                    embedding_dim = len(embeddings[0]['embedding']) if embeddings else 0
                    file_size_mb = txt_file.stat().st_size / (1024**2)
                    
                    summary_data.append({
                        'model': model_name,
                        'gene': gene_name,
                        'num_sequences': num_sequences,
                        'embedding_dimension': embedding_dim,
                        'file_size_mb': round(file_size_mb, 2)
                    })
                    
                except Exception as e:
                    print(f"Error reading {txt_file}: {e}")
    
    # Create DataFrame and save
    df = pd.DataFrame(summary_data)
    df.to_csv('embedding_summary.csv', index=False)
    
    print("Summary report saved to 'embedding_summary.csv'")
    print(f"Total files processed: {len(summary_data)}")
    
    # Display summary statistics
    if not df.empty:
        print("\nSummary Statistics:")
        print(f"Models: {df['model'].nunique()}")
        print(f"Genes: {df['gene'].nunique()}")
        print(f"Embedding dimensions: {df['embedding_dimension'].unique()}")
        print(f"Total file size: {df['file_size_mb'].sum():.2f} MB")
    
    return df

# Uncomment lines below to run post-processing:
# convert_embeddings_to_numpy()
# summary_df = create_summary_report()